# Data Overview & Analysis

**Complete Data Pipeline Visualization**

Raw Chunks → SFT Candidates → DPO Pairs

Covers:
- ✅ Single-dataset pipeline (Asas Al-Balagha only)
- ✅ Multi-dataset pipeline (All sources combined)
- ✅ Statistics, samples, quality metrics, and distributions

## Setup

In [ ]:
import os
import sys
import json
from pathlib import Path
from collections import Counter, defaultdict

# Initialize project path (portable - works on any computer)
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    # Fallback: go up one level if src not in current directory
    PROJECT_ROOT = PROJECT_ROOT.parent
    if not (PROJECT_ROOT / 'src').exists():
        raise FileNotFoundError('Could not find src/ directory. Make sure you run this notebook from the project root.')

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from sft import load_chunks
from sft.schema import SFTSample

print(f'✓ Project: {PROJECT_ROOT.name}')

## 1. All Available Datasets

In [ ]:
print('\n=== ALL SOURCE DATASETS ===')

datasets = {
    'asas_albalagha': {
        'path': PROJECT_ROOT / 'data' / 'processed' / 'chunks_asas_albalagha.jsonl',
        'name': 'Asas Al-Balagha',
        'dialect': 'Classical Arabic',
    },
    'najdi_popular': {
        'path': PROJECT_ROOT / 'data' / 'processed' / 'chunks_najdi_popular.jsonl',
        'name': 'Najdi Popular',
        'dialect': 'Modern Saudi Dialect',
    },
}

dataset_stats = {}
for key, config in datasets.items():
    exists = config['path'].exists()
    count = sum(1 for _ in open(config['path'])) if exists else 0
    size_mb = config['path'].stat().st_size / (1024*1024) if exists else 0
    
    status = '✓' if exists else '✗'
    print(f'{status} {config["name"]:25s} {count:4d} chunks ({size_mb:5.1f} MB) - {config["dialect"]}')
    
    if exists:
        dataset_stats[key] = {'count': count, 'size_mb': size_mb}

total_chunks = sum(s['count'] for s in dataset_stats.values())
total_size = sum(s['size_mb'] for s in dataset_stats.values())
print(f'\nTotal: {total_chunks} chunks ({total_size:.1f} MB)')

## 2. Training Data - Single Dataset (Asas Al-Balagha Only)

In [ ]:
print('\n=== SINGLE-DATASET PIPELINE (Asas Al-Balagha) ===')

# Single dataset paths
SINGLE_DATASET = {
    'chunks': PROJECT_ROOT / 'data' / 'processed' / 'chunks_asas_albalagha.jsonl',
    'sft_candidates': PROJECT_ROOT / 'data' / 'generated' / 'sft' / 'candidates.jsonl',
    'sft_accepted': PROJECT_ROOT / 'data' / 'generated' / 'sft' / 'accepted.jsonl',
    'dpo_candidates': PROJECT_ROOT / 'data' / 'generated' / 'dpo' / 'candidates.jsonl',
}

single_stats = {}
for key, path in SINGLE_DATASET.items():
    if path.exists():
        count = sum(1 for _ in open(path))
        size_mb = path.stat().st_size / (1024*1024)
        single_stats[key] = count
        label = key.replace('_', ' ').title()
        print(f'  ✓ {label:20s} {count:6,d} {"pairs" if "dpo" in key else "samples":10s} ({size_mb:5.1f} MB)')

print(f'\n  Summary: 394 chunks → {single_stats.get("sft_candidates", 0):,} SFT → {single_stats.get("dpo_candidates", 0):,} DPO')

## 3. Training Data - Multi-Dataset (All Sources)

In [ ]:
print('\n=== MULTI-DATASET PIPELINE (All Sources) ===')

# Multi-dataset paths
MULTI_DATASET = {
    'sft_candidates': PROJECT_ROOT / 'data' / 'generated' / 'sft' / 'candidates_multi.jsonl',
    'sft_accepted': PROJECT_ROOT / 'data' / 'generated' / 'sft' / 'accepted_multi.jsonl',
    'dpo_candidates': PROJECT_ROOT / 'data' / 'generated' / 'dpo' / 'candidates_multi.jsonl',
}

multi_stats = {}
has_multi = False

for key, path in MULTI_DATASET.items():
    if path.exists():
        has_multi = True
        count = sum(1 for _ in open(path))
        size_mb = path.stat().st_size / (1024*1024)
        multi_stats[key] = count
        label = key.replace('_', ' ').title()
        print(f'  ✓ {label:20s} {count:6,d} {"pairs" if "dpo" in key else "samples":10s} ({size_mb:5.1f} MB)')

if has_multi:
    total_multi_chunks = dataset_stats.get('asas_albalagha', {}).get('count', 0) + dataset_stats.get('najdi_popular', {}).get('count', 0)
    print(f'\n  Summary: {total_multi_chunks} chunks → {multi_stats.get("sft_candidates", 0):,} SFT → {multi_stats.get("dpo_candidates", 0):,} DPO')
else:
    print('  ⚠️  Multi-dataset files not yet generated')
    print('  Run: notebooks/multi_dataset_pipeline.ipynb')

## 4. Pipeline Comparison

In [ ]:
print('\n=== PIPELINE COMPARISON ===')
print(f"\n{'Metric':<30s} {'Single-Dataset':>20s} {'Multi-Dataset':>20s}")
print('─' * 72)

# Source chunks
single_chunks = dataset_stats.get('asas_albalagha', {}).get('count', 0)
multi_chunks = sum(s.get('count', 0) for s in dataset_stats.values())
print(f"{'Source chunks':<30s} {single_chunks:>20,d} {multi_chunks:>20,d}")

# Source datasets
print(f"{'Source datasets':<30s} {'1 (Classical)':>20s} {'2 (Classical+Modern)':>20s}")

# SFT samples
sft_single = single_stats.get('sft_candidates', 0)
sft_multi = multi_stats.get('sft_candidates', 0) if has_multi else 0
print(f"{'SFT Candidates':<30s} {sft_single:>20,d} {sft_multi:>20,d}")

# SFT accepted
sft_acc_single = single_stats.get('sft_accepted', 0)
sft_acc_multi = multi_stats.get('sft_accepted', 0) if has_multi else 0
print(f"{'SFT Accepted':<30s} {sft_acc_single:>20,d} {sft_acc_multi:>20,d}")

# DPO pairs
dpo_single = single_stats.get('dpo_candidates', 0)
dpo_multi = multi_stats.get('dpo_candidates', 0) if has_multi else 0
print(f"{'DPO Pairs':<30s} {dpo_single:>20,d} {dpo_multi:>20,d}")

# Expansion rates
sft_exp_single = sft_single / single_chunks if single_chunks > 0 else 0
sft_exp_multi = sft_multi / multi_chunks if multi_chunks > 0 else 0
print(f"{'SFT expansion rate':<30s} {sft_exp_single:>20.1f}x {sft_exp_multi:>20.1f}x")

# DPO conversion
dpo_conv_single = dpo_single / sft_acc_single * 100 if sft_acc_single > 0 else 0
dpo_conv_multi = dpo_multi / sft_acc_multi * 100 if sft_acc_multi > 0 else 0
print(f"{'DPO conversion rate':<30s} {dpo_conv_single:>19.1f}% {dpo_conv_multi:>19.1f}%")

## 5. Data Quality Insights

In [ ]:
print('\n=== DATA QUALITY & BENEFITS ===')

print('\n✓ Single-Dataset (Asas Al-Balagha Only):')
print('  • Focused on classical Arabic')
print('  • 394 chunks, 4,645 SFT samples')
print('  • 3,000 DPO preference pairs')
print('  • Limited dialect diversity')
print('  • Single-document constraint (no proper splits)')

if has_multi:
    print('\n✓ Multi-Dataset (All Sources):')
    print(f'  • {total_multi_chunks} total chunks')
    print('  • Classical + Modern Saudi dialects')
    print(f'  • {sft_multi:,} SFT samples')
    print(f'  • {dpo_multi:,} DPO pairs')
    print('  • Better generalization')
    print('  • Proper document-level train/val/test splits possible')
else:
    print('\n✓ Multi-Dataset (All Sources) - Not Yet Generated:')
    print('  • Would combine Asas Al-Balagha + Najdi Popular')
    print('  • Enable document-level splits')
    print('  • Improve dialect diversity')
    print('  • Better generalization to real-world Arabic')
    print('\n  → Run: notebooks/multi_dataset_pipeline.ipynb')

print('\n✓ All data available for training!')